# Camada Bronze - Marketing Funnel by Olist + IPCA (Bruno)

Carrega os ficheiros originais para o HDFS e cria as tabelas Bronze em formato Delta, na stack comum do grupo.
Os dados ficam tal como vêm da fonte (todas as colunas como texto, com os nomes originais),
com os metadados de ingestão `_data_ingestao` e `_ficheiro_origem`.

| Dataset | Ficheiro | Tabela Bronze |
|---|---|---|
| Marketing Funnel (principal) | olist_marketing_qualified_leads_dataset.csv | bronze.funil_mql |
| Marketing Funnel (principal) | olist_closed_deals_dataset.csv | bronze.funil_closed_deals |
| Brazil SELIC and Inflation (IPCA) Rates History (complementar) | IBGE_IPCA.csv | bronze.ipca |

Fontes: Kaggle `olistbr/marketing-funnel-olist` e `hssiqueira/brazil-interest-rate-history-selic`.
A Bronze guarda todas as datas; o filtro do intervalo do grupo (junho a dezembro de 2017) é feito na Silver.

In [1]:
ficheiros = {
    "/home/jovyan/work/data/bruno/olist_marketing_qualified_leads_dataset.csv": "/projeto/bronze/bruno/olist_marketing_qualified_leads_dataset.csv",
    "/home/jovyan/work/data/bruno/olist_closed_deals_dataset.csv": "/projeto/bronze/bruno/olist_closed_deals_dataset.csv",
    "/home/jovyan/work/data/bruno/IBGE_IPCA.csv": "/projeto/bronze/bruno/IBGE_IPCA.csv",
}

In [2]:
# Copiar os ficheiros originais para a zona Bronze do HDFS (full load: apaga a versão anterior e volta a carregar)
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

for from_path, to_path in ficheiros.items():
    client.delete(to_path)
    client.upload(to_path, from_path)
    print(to_path, client.status(to_path)["length"], "bytes")

/projeto/bronze/bruno/olist_marketing_qualified_leads_dataset.csv 703836 bytes
/projeto/bronze/bruno/olist_closed_deals_dataset.csv 171352 bytes
/projeto/bronze/bruno/IBGE_IPCA.csv 5206 bytes


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Bronze - Marketing Funnel + IPCA (Bruno)") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Os nomes originais das colunas têm espaços e caracteres especiais;
# o column mapping do Delta permite guardá-los tal como vêm da fonte
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")
spark.conf.set("spark.databricks.delta.properties.defaults.minReaderVersion", "2")
spark.conf.set("spark.databricks.delta.properties.defaults.minWriterVersion", "5")

In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db/'")

def gravar_bronze(df, tabela):
    """Acrescenta os metadados de ingestão e grava a tabela Delta tal como veio da fonte."""
    df = df.withColumn("_data_ingestao", F.current_timestamp()) \
           .withColumn("_ficheiro_origem", F.input_file_name())
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/bronze.db/{tabela}/") \
        .saveAsTable(f"bronze.{tabela}")
    print(f"bronze.{tabela}: {spark.table(f'bronze.{tabela}').count()} linhas, {len(df.columns) - 2} colunas originais")

def ler_csv(hdfs_path, **opcoes):
    """Lê um CSV do HDFS com todas as colunas como texto (sem inferSchema)."""
    leitor = spark.read.option("header", "true").option("multiLine", "true").option("escape", '"')
    for k, v in opcoes.items():
        leitor = leitor.option(k, v)
    return leitor.csv("hdfs://hdfs-nn:9000" + hdfs_path)

In [5]:
gravar_bronze(ler_csv("/projeto/bronze/bruno/olist_marketing_qualified_leads_dataset.csv"), "funil_mql")
gravar_bronze(ler_csv("/projeto/bronze/bruno/olist_closed_deals_dataset.csv"), "funil_closed_deals")
gravar_bronze(ler_csv("/projeto/bronze/bruno/IBGE_IPCA.csv"), "ipca")

bronze.funil_mql: 8000 linhas, 4 colunas originais
bronze.funil_closed_deals: 842 linhas, 14 colunas originais
bronze.ipca: 394 linhas, 2 colunas originais


## Verificação

As tabelas Bronze têm de ter o mesmo número de linhas dos ficheiros originais (contadas aqui diretamente nos ficheiros).

In [6]:
import csv
def linhas_csv(caminho):
    with open(caminho, encoding="utf-8", newline="") as f:
        return sum(1 for _ in csv.reader(f)) - 1   # sem o cabeçalho

origem = {
    "funil_mql": "/home/jovyan/work/data/bruno/olist_marketing_qualified_leads_dataset.csv",
    "funil_closed_deals": "/home/jovyan/work/data/bruno/olist_closed_deals_dataset.csv",
    "ipca": "/home/jovyan/work/data/bruno/IBGE_IPCA.csv",
}
for tabela, caminho in origem.items():
    n, real = linhas_csv(caminho), spark.table(f"bronze.{tabela}").count()
    print(f"{tabela:20s} ficheiro {n:>6}  bronze {real:>6}  {'OK' if n == real else 'DIFERENTE'}")
    assert n == real

funil_mql            ficheiro   8000  bronze   8000  OK
funil_closed_deals   ficheiro    842  bronze    842  OK
ipca                 ficheiro    394  bronze    394  OK


In [7]:
spark.table("bronze.funil_mql").show(3, truncate=False)
spark.table("bronze.funil_closed_deals").select("mql_id", "seller_id", "won_date", "business_segment", "lead_type").show(3, truncate=False)
spark.table("bronze.ipca").show(3, truncate=False)

+--------------------------------+------------------+--------------------------------+--------------+--------------------------+------------------------------------------------------------------------------------+
|mql_id                          |first_contact_date|landing_page_id                 |origin        |_data_ingestao            |_ficheiro_origem                                                                    |
+--------------------------------+------------------+--------------------------------+--------------+--------------------------+------------------------------------------------------------------------------------+
|dac32acd4db4c29c230538b72f8dd87d|2018-02-01        |88740e65d5d6b056e0cda098e1ea6313|social        |2026-11-12 18:59:36.594773|hdfs://hdfs-nn:9000/projeto/bronze/bruno/olist_marketing_qualified_leads_dataset.csv|
|8c18d1de7f67e60dbd64e3c07d7e9d5d|2017-10-20        |007f9098284a86ee80ddeb25d53e0af8|paid_search   |2026-11-12 18:59:36.594773|hdfs://hdfs-nn:9

In [8]:
spark.stop()